# HYPERPARAMETER TUNING FOR EVERY STOCK AND FOR EVERY TIMEFRAME

In this notebook, we are specially focusing on finding the best parameters for every possible combination of stock symbol and timeframe. as we cant fit all stocks on a single result of hyperparameter tuning. 



Note: This provides reasonable params as we have artificially injected the anomalies. These are not the best optimal params as this would need labelled data.

## Imports

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

from src.utils.timeframes import VALID_TIMEFRAMES, WINDOW_MAP
from src.utils.load import load_json
from src.utils.paths import HYPERPARAMS, ARTIFACTS, CONFIG
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from src.models.isolation_forest import IsolationForest
from src.models.zscore import zscore
from src.models.dbscan import DBSCAN
from itertools import product
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)
import matplotlib.pyplot as plt
from src.components.anomaly_detection import train_model
from src.components.data_loader import load_data
from src.components.preprocessing import preprocess
from src.components.feature_engineering import build_features
from src.components.scaling import scale_features
import json
from src.utils.load import load_config,load_json
from datetime import datetime
from dateutil.relativedelta import relativedelta



import warnings

warnings.filterwarnings("ignore")

## Inject Anomalies for Evaluation

| Anomaly Type      | Impact on 4D Space                                                |
| ----------------- | ----------------------------------------------------------------- |
| pump/dump         | close ↑, volume → (basic)                                         |
| volume_spike      | volume ↑↑, returns/volatility normal                              |
| inverse_volume    | volume ↓ BUT close ↑↑ (breaks correlation)                        |
| volatility_crush  | volatility ↓↓ (opposite of expected)                              |
| gap_open          | returns ↑↑ on single point (isolated)                             |
| wick_spike        | volatility ↑↑ but returns ≈ 0 (decouples returns from volatility) |
| correlation_break | volume ↑↑↑ but returns ≈ 0 (strongest break)                      |


In [ ]:
import numpy as np
import pandas as pd

def inject_anomalies(df, features):
    """
    Inject synthetic anomalies into OHLCV data for model evaluation.
    
    ANOMALY TYPES (realistic market scenarios):
    ============================================
    - pump (25%): Price spike up suddenly
    - dump (25%): Price spike down suddenly
    - volume_spike (20%): Extreme volume with normal price movement
    - inverse_volume (15%): Price moves significantly BUT volume stays low (suspicious)
    - volatility_crush (15%): Candle very tight with low volume (dead market)
    
    CHARACTERISTICS:
    - Each anomaly lasts 3-8 candles (multi-candle realistic moves)
    - Total injected anomalies ≈ 2% of dataset × avg_window_length (5.5)
    - Results in 6-11% of rows marked as anomalous (varies by timeframe)
    
    DESIGN FOR DETECTION MODELS:
    - Isolation Forest: Anomalies scattered across 4D space, require deep isolation
    - DBSCAN: Anomalies create density clusters at edges of feature space
    - Z-Score: Each anomaly type violates at least one univariate threshold
    
    Args:
        df (pd.DataFrame): 
            OHLCV data with columns: open, high, low, close, volume
            Must have 'True_Anomaly' column initialized to 0
        features (list):
            Feature columns to validate after injection
            Expected: ['close', 'volume', 'returns', 'volatility']
    
    Returns:
        pd.DataFrame:
            Original df + injected anomalies with recalculated returns/volatility
            Rows with NaN in features are removed
    """
    
    # ============ INJECTION PARAMETERS ============
    fraction = 0.02  # Target: 2% of data points should be anomalies
    price_range = (0.05, 0.40)  # Price movement: 5-40%
    vol_multiplier = (4, 12)  # Volume spike: 4-12x normal
    window_size = (3, 8)  # Anomaly duration: 3-8 candles
    
    df_injected = df.copy()
    df_injected["True_Anomaly"] = 0
    
    n = len(df_injected)
    
    # ============ CALCULATE NUMBER OF ANOMALY WINDOWS ============
    # Since each anomaly window lasts ~5.5 candles on average,
    # we need fewer starting points to hit ~2% target
    expected_window_length = (window_size[0] + window_size[1]) / 2  # 5.5
    k = max(1,int((n * fraction) / expected_window_length))   # Adjust for window duration
    
    # Select random non-overlapping start indices
    idxs = np.random.choice(range(n - max(window_size)), size=k, replace=False)
    
    # ============ INJECT ANOMALIES ============
    for start_idx in idxs:
        # Random duration for this anomaly window
        duration = np.random.randint(*window_size)
        
        # Choose anomaly type with probabilities optimized for detection difficulty
        anomaly_type = np.random.choice(
            ["pump", "dump", "volume_spike", "inverse_volume", "volatility_crush"],
            p=[0.25, 0.25, 0.20, 0.15, 0.15]
        )
        
        # Apply same anomaly type to entire window (realistic)
        for i in range(start_idx, start_idx + duration):
            # Mark row as anomalous
            df_injected.iloc[i, df_injected.columns.get_loc("True_Anomaly")] = 1
            
            if anomaly_type == "pump":
                # Price jumps up suddenly
                factor = np.random.uniform(*price_range)
                factor += np.random.normal(0, 0.003)  # Add small noise
                for col in ["open", "high", "low", "close"]:
                    col_idx = df_injected.columns.get_loc(col)
                    df_injected.iloc[i, col_idx] *= 1 + factor
                    
            elif anomaly_type == "dump":
                # Price crashes down suddenly
                factor = np.random.uniform(*price_range) * -1
                factor += np.random.normal(0, 0.003)
                for col in ["open", "high", "low", "close"]:
                    col_idx = df_injected.columns.get_loc(col)
                    df_injected.iloc[i, col_idx] *= 1 + factor
                    
            elif anomaly_type == "volume_spike":
                # Volume spikes WITHOUT corresponding price move
                # (detected as outlier in [volume] space)
                col_idx = df_injected.columns.get_loc("volume")
                df_injected.iloc[i, col_idx] *= np.random.uniform(*vol_multiplier)
                
            elif anomaly_type == "inverse_volume":
                # Price moves BIG but volume is suspiciously LOW
                # (breaks volume-price correlation)
                col_idx = df_injected.columns.get_loc("volume")
                df_injected.iloc[i, col_idx] *= np.random.uniform(0.05, 0.2)  # Very low
                # But price moves a lot
                factor = np.random.uniform(0.10, 0.35)
                factor *= np.random.choice([-1, 1])
                for col in ["open", "high", "low", "close"]:
                    col_idx = df_injected.columns.get_loc(col)
                    df_injected.iloc[i, col_idx] *= 1 + factor
                    
            elif anomaly_type == "volatility_crush":
                # Candle becomes EXTREMELY tight (low volatility)
                # BUT volume also drops (unnatural combination)
                close = df_injected.iloc[i]["close"]
                # Tiny range: 0.00005 to 0.0005 (0.005% to 0.05%)
                epsilon = np.random.uniform(0.00005, 0.0005)
                for col in ["open", "high", "low", "close"]:
                    col_idx = df_injected.columns.get_loc(col)
                    df_injected.iloc[i, col_idx] = close * (1 + np.random.uniform(-epsilon, epsilon))
                # Also reduce volume
                col_idx = df_injected.columns.get_loc("volume")
                df_injected.iloc[i, col_idx] *= np.random.uniform(0.3, 0.6)
    
    # ============ POST-INJECTION PROCESSING ============
    # Convert anomaly column to integer
    df_injected["True_Anomaly"] = df_injected["True_Anomaly"].astype(int)
    
    # Recalculate features AFTER injecting anomalies
    # This ensures anomalies affect returns & volatility calculations
    df_injected["returns"] = df_injected["close"].pct_change()
    df_injected["volatility"] = df_injected["returns"].rolling(window=20).std()
    
    # Clean: replace inf/-inf with NaN
    df_injected.replace([np.inf, -np.inf], np.nan, inplace=True)
    
    # Remove rows with NaN in required features
    df_injected.dropna(subset=features, inplace=True)
    
    return df_injected

## Tuning Functions for different models

In [3]:
def quick_tune_isolation_forest(X, y_true, verbose=True):
    results = []

    param_grid = {
        "n_estimators": [100, 200, 300],
        "contamination": [0.005, 0.01, 0.02, 0.05],
    }

    combinations = list(
        product(param_grid["n_estimators"], param_grid["contamination"])
    )

    if verbose:
        print(f"Testing {len(combinations)} Isolation Forest combinations...")

    for n_est, contam in combinations:
        try:

            model = IsolationForest(
                n_trees=n_est, contamination=contam, random_state=42
            )

            predictions = model.fit_predict(X)
            predictions_binary = (predictions == -1).astype(int)

            # Skip if no anomalies
            if predictions_binary.sum() == 0:
                continue

            f1 = f1_score(y_true, predictions_binary, zero_division=0)
            precision = precision_score(y_true, predictions_binary, zero_division=0)
            recall = recall_score(y_true, predictions_binary, zero_division=0)

            results.append(
                {
                    "n_estimators": n_est,
                    "contamination": contam,
                    "f1_score": f1,
                    "precision": precision,
                    "recall": recall,
                }
            )

        except Exception as e:
            if verbose:
                print(f"  Error with n_est={n_est}, contam={contam}: {e}")

    if not results:
        # Return empty dataframe with proper structure
        return pd.DataFrame(
            {
                "n_estimators": [200],
                "contamination": [0.01],
                "f1_score": [0.0],
                "precision": [0.0],
                "recall": [0.0],
            }
        )

    results_df = pd.DataFrame(results).sort_values("f1_score", ascending=False)

    if verbose:
        print(f"\n✓ Best F1-Score: {results_df.iloc[0]['f1_score']:.4f}")
        print(
            f"✓ Best Parameters: n_estimators={int(results_df.iloc[0]['n_estimators'])}, "
            f"contamination={results_df.iloc[0]['contamination']:.4f}\n"
        )

    return results_df


def quick_tune_dbscan(X, y_true, verbose=True):
    """Quick tuning of DBSCAN."""

    results = []

    param_grid = {"eps": [0.5, 1.0, 1.5, 2.0], "min_pts": [5, 10, 15]}

    combinations = list(product(param_grid["eps"], param_grid["min_pts"]))

    if verbose:
        print(f"Testing {len(combinations)} DBSCAN combinations...")

    for eps, min_pts in combinations:
        try:
            model = DBSCAN(eps=eps, min_pts=min_pts)
            predictions = model.fit_predict(X)

            # FIX: Handle case where DBSCAN returns boolean or scalar
            if isinstance(predictions, (bool, np.bool_)):
                if verbose:
                    print(
                        f"  Error with eps={eps}, min_pts={min_pts}: Invalid return type (boolean)"
                    )
                continue

            # Convert to numpy array if needed
            predictions = np.asarray(predictions)

            # Check if it's a scalar
            if predictions.ndim == 0:
                if verbose:
                    print(
                        f"  Error with eps={eps}, min_pts={min_pts}: Scalar return value"
                    )
                continue

            # Now safe to convert to int
            predictions_binary = (predictions == -1).astype(int)

            if predictions_binary.sum() == 0:
                continue

            f1 = f1_score(y_true, predictions_binary, zero_division=0)
            precision = precision_score(y_true, predictions_binary, zero_division=0)
            recall = recall_score(y_true, predictions_binary, zero_division=0)

            results.append(
                {
                    "eps": eps,
                    "min_pts": min_pts,
                    "f1_score": f1,
                    "precision": precision,
                    "recall": recall,
                }
            )

        except Exception as e:
            if verbose:
                print(f"  Error with eps={eps}, min_pts={min_pts}: {str(e)[:60]}")

    if not results:
        # Return empty dataframe with proper structure
        return pd.DataFrame(
            {
                "eps": [1.0],
                "min_pts": [5],
                "f1_score": [0.0],
                "precision": [0.0],
                "recall": [0.0],
            }
        )

    results_df = pd.DataFrame(results).sort_values("f1_score", ascending=False)

    if verbose:
        print(f"\n✓ Best F1-Score: {results_df.iloc[0]['f1_score']:.4f}")
        print(
            f"✓ Best Parameters: eps={results_df.iloc[0]['eps']:.2f}, "
            f"min_pts={int(results_df.iloc[0]['min_pts'])}\n"
        )

    return results_df


def quick_tune_z_score(df, y_true, feature="close", verbose=True):

    results = []
    thresholds = np.arange(1.5, 4.5, 0.5)

    if verbose:
        print(f"Testing {len(thresholds)} Z-Score thresholds...")

    z_scores = zscore(df[feature].dropna())

    for threshold in thresholds:
        predictions_binary = (np.abs(z_scores) > threshold).astype(int)

        if predictions_binary.sum() == 0:
            continue

        # Need to align with y_true
        valid_idx = df[feature].notna()
        y_valid = y_true[valid_idx]

        if len(y_valid) != len(predictions_binary):
            continue

        f1 = f1_score(y_valid, predictions_binary, zero_division=0)
        precision = precision_score(y_valid, predictions_binary, zero_division=0)
        recall = recall_score(y_valid, predictions_binary, zero_division=0)

        results.append(
            {
                "threshold": threshold,
                "f1_score": f1,
                "precision": precision,
                "recall": recall,
                "n_anomalies": predictions_binary.sum(),
            }
        )

    if not results:
        # Return empty dataframe with proper structure
        return pd.DataFrame(
            {
                "threshold": [3.0],
                "f1_score": [0.0],
                "precision": [0.0],
                "recall": [0.0],
                "n_anomalies": [0],
            }
        )

    results_df = pd.DataFrame(results).sort_values("f1_score", ascending=False)

    if verbose:
        print(f"\n✓ Best F1-Score: {results_df.iloc[0]['f1_score']:.4f}")
        print(f"✓ Best Threshold: {results_df.iloc[0]['threshold']:.2f}\n")

    return results_df



## Visualize hyperparameters

In [4]:

# visualize the results


def plot_if_tuning(results_df):
    """Plot Isolation Forest tuning results."""
    try:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        for n_est in results_df["n_estimators"].unique():
            subset = results_df[results_df["n_estimators"] == n_est].sort_values(
                "contamination"
            )
            axes[0].plot(
                subset["contamination"],
                subset["f1_score"],
                marker="o",
                label=f"n_est={int(n_est)}",
            )

        axes[0].set_xlabel("Contamination")
        axes[0].set_ylabel("F1-Score")
        axes[0].set_title("Isolation Forest: F1-Score vs Contamination")
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)

        # Top results
        top_10 = results_df.head(10).sort_values("f1_score")
        axes[1].barh(range(len(top_10)), top_10["f1_score"], color="steelblue")
        labels = [
            f"n_est={int(row['n_estimators'])}, cont={row['contamination']:.3f}"
            for _, row in top_10.iterrows()
        ]
        axes[1].set_yticks(range(len(top_10)))
        axes[1].set_yticklabels(labels, fontsize=9)
        axes[1].set_xlabel("F1-Score")
        axes[1].set_title("Top 10 Parameter Combinations")

        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not plot Isolation Forest results: {e}")


def plot_dbscan_tuning(results_df):
    """Plot DBSCAN tuning results as heatmap."""
    try:
        import seaborn as sns

        # Check if we have enough data to pivot
        if len(results_df) < 2:
            print("Not enough DBSCAN results to plot")
            return

        # Pivot for heatmap
        pivot_table = results_df.pivot_table(
            index="min_pts", columns="eps", values="f1_score", aggfunc="mean"
        )

        if pivot_table.empty:
            print("Could not create pivot table for DBSCAN")
            return

        plt.figure(figsize=(8, 6))
        sns.heatmap(
            pivot_table,
            annot=True,
            fmt=".3f",
            cmap="RdYlGn",
            cbar_kws={"label": "F1-Score"},
        )
        plt.title("DBSCAN: F1-Score Heatmap")
        plt.xlabel("eps")
        plt.ylabel("min_pts")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not plot DBSCAN results: {e}")


def plot_zscore_tuning(results_df):
    """Plot Z-Score tuning results."""
    try:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        axes[0].plot(
            results_df["threshold"],
            results_df["f1_score"],
            marker="o",
            color="red",
            linewidth=2,
            markersize=8,
        )
        axes[0].set_xlabel("Threshold")
        axes[0].set_ylabel("F1-Score")
        axes[0].set_title("Z-Score: F1-Score vs Threshold")
        axes[0].grid(True, alpha=0.3)

        axes[1].plot(
            results_df["threshold"],
            results_df["n_anomalies"],
            marker="s",
            color="purple",
            linewidth=2,
            markersize=8,
        )
        axes[1].set_xlabel("Threshold")
        axes[1].set_ylabel("Number of Anomalies")
        axes[1].set_title("Z-Score: Anomaly Count vs Threshold")
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Could not plot Z-Score results: {e}")


## Run the functions

In [5]:
def run_hyperparameter_tuning(df, features, symbol, timeframe,verbose=True):

    print("\n" + "=" * 60)
    print(f"HYPERPARAMETER TUNING FOR ANOMALY DETECTION for {symbol} at {timeframe}")
    print("=" * 60 + "\n")

    # Prepare data

    X = scale_features(df,features)

    y_true = df["True_Anomaly"].values

    best_params = {}

    # 1. ISOLATION FOREST

    print(" TUNING ISOLATION FOREST")
    print("-" * 60)

    results_if = quick_tune_isolation_forest(X, y_true, verbose=verbose)

    best_params["isolation_forest"] = {
        "n_estimators": int(results_if.iloc[0]["n_estimators"]),
        "contamination": float(results_if.iloc[0]["contamination"]),
    }

    if verbose and len(results_if) > 0:
        print("Results:")
        print(
            results_if[
                ["n_estimators", "contamination", "precision", "recall", "f1_score"]
            ].head()
        )

    # try:
        # plot_if_tuning(results_if)
    # except Exception as e:
    #     print(f"Could not plot Isolation Forest: {e}")

    # 2. DBSCAN

    print("\n TUNING DBSCAN")
    print("-" * 60)

    results_dbscan = quick_tune_dbscan(X, y_true, verbose=verbose)

    best_params["dbscan"] = {
        "eps": float(results_dbscan.iloc[0]["eps"]),
        "min_pts": int(results_dbscan.iloc[0]["min_pts"]),
    }

    if verbose and len(results_dbscan) > 0:
        print("Results:")
        cols_to_show = [
            c
            for c in ["eps", "min_pts", "precision", "recall", "f1_score"]
            if c in results_dbscan.columns
        ]
        print(results_dbscan[cols_to_show].head())

    # try:
        # plot_dbscan_tuning(results_dbscan)
    # except Exception as e:
    #     print(f"Could not plot DBSCAN: {e}")

    # 3. Z-SCORE

    print("\n TUNING Z-SCORE")
    print("-" * 60)

    results_zscore = quick_tune_z_score(df, y_true, verbose=verbose)

    best_params["z_score"] = {"threshold": float(results_zscore.iloc[0]["threshold"])}

    if verbose and len(results_zscore) > 0:
        print("Results:")
        print(
            results_zscore[
                ["threshold", "precision", "recall", "f1_score", "n_anomalies"]
            ]
        )

    # try:
        # plot_zscore_tuning(results_zscore)
    # except Exception as e:
    #     print(f"Could not plot Z-Score: {e}")

    # Summary
    print("\n" + "=" * 60)
    print("SUMMARY OF BEST PARAMETERS")
    print("=" * 60)

    for method, params in best_params.items():
        print(f"\n{method.upper()}:")
        for param, value in params.items():
            print(f"  {param}: {value}")



    return best_params, {
        "if": results_if,
        "dbscan": results_dbscan,
        "zscore": results_zscore,
    }


if __name__ == "__main__":
    print("This module is ready to be imported and used in your notebook.")
    print("\nUsage:")
    print(
        "  best_params = run_hyperparameter_tuning(df, features=['close', 'volume', 'returns', 'volatility'])"
    )

This module is ready to be imported and used in your notebook.

Usage:
  best_params = run_hyperparameter_tuning(df, features=['close', 'volume', 'returns', 'volatility'])


## Tune hyperparamters for every possible combination

In [6]:
config = load_config(CONFIG / "config.yaml")
stocks = sorted(load_json(ARTIFACTS / "allowed_symbols.json"))
features = config["features"]


for stock in stocks:

    output_path = HYPERPARAMS / f"{stock}.json"

    # resume support
    # if output_path.exists():
    #     print(f"Skipping {stock}")
    #     continue
    
    best_params_stock = {}



    for timeframe in VALID_TIMEFRAMES:

        days = WINDOW_MAP[timeframe]
        data = load_data(
            symbol=stock,
            start_date=(datetime.now() - relativedelta(days = days)).date(),
            end_date=datetime.now().date()
        )
        

        clean_data = preprocess(data,timeframe)

        df_injected = inject_anomalies(clean_data,features)

        print(stock, timeframe)
        print("raw:", len(data))
        print("clean:", len(clean_data))
        print(f"Injected Anomalies: {len(df_injected[df_injected['True_Anomaly'] == 1])}")

        best_params, all_results = run_hyperparameter_tuning(df_injected, features,stock,timeframe)

        z_score_threshold = best_params["z_score"]["threshold"]

        n_estimators = best_params["isolation_forest"]["n_estimators"]
        contamination = best_params["isolation_forest"]["contamination"]

        epsilon = best_params["dbscan"]["eps"]
        min_samples = best_params["dbscan"]["min_pts"]

        best_params_stock[timeframe] = best_params

    


    
    with open(output_path,"w") as f:
        json.dump(best_params_stock,f,indent=2);
    
    break

API 1min
raw: 12807
clean: 2883
Injected Anomalies: 43

HYPERPARAMETER TUNING FOR ANOMALY DETECTION for API at 1min

 TUNING ISOLATION FOREST
------------------------------------------------------------
Testing 12 Isolation Forest combinations...

✓ Best F1-Score: 0.5556
✓ Best Parameters: n_estimators=100, contamination=0.0100

Results:
    n_estimators  contamination  precision    recall  f1_score
1            100           0.01   0.689655  0.465116  0.555556
5            200           0.01   0.689655  0.465116  0.555556
9            300           0.01   0.689655  0.465116  0.555556
2            100           0.02   0.410714  0.534884  0.464646
10           300           0.02   0.396552  0.534884  0.455446

 TUNING DBSCAN
------------------------------------------------------------
Testing 12 DBSCAN combinations...

✓ Best F1-Score: 0.6133
✓ Best Parameters: eps=2.00, min_pts=15

Results:
    eps  min_pts  precision    recall  f1_score
11  2.0       15    0.71875  0.534884  0.613333
